# Week 8: NLP Foundations & Text Embeddings
## Notebook 11: Tokenization Mechanics: BPE vs WordPiece

### 1. Conceptual Intuition & Theoretical Background
Subword tokenizers bridge the gap between word-level models (which suffer from out-of-vocabulary words) and character-level models (which produce very long sequences).

**1. Byte-Pair Encoding (BPE, Sennrich et al., 2016):**
- **Merge Criterion:** Deterministically merges the **most frequent symbol pair**:
$$\arg\max_{(u, v)} \text{Count}(u, v)$$
- Used in GPT-2, GPT-4, LLaMA, and RoBERTa.

**2. WordPiece (Schuster & Nakajima, 2012):**
- **Merge Criterion:** Maximizes the **likelihood ratio** of the language model, scoring pairs by how much they exceed independent probability:
$$\text{Score}(u, v) = \frac{\text{Count}(u, v)}{\text{Count}(u) \times \text{Count}(v)}$$
- Used in BERT and Google Translate.

---

### 2. Scratch Python Implementation: BPE vs WordPiece Merge Logic

In [ ]:
from collections import Counter, defaultdict

def get_stats(corpus):
    pairs = defaultdict(int)
    for word, freq in corpus.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

def bpe_merge_step(corpus):
    """Finds best pair by pure frequency."""
    pairs = get_stats(corpus)
    if not pairs:
        return None, corpus
    best_pair = max(pairs, key=pairs.get)
    
    # Execute merge
    new_corpus = {}
    pattern = ' '.join(best_pair)
    replacement = ''.join(best_pair)
    for word in corpus:
        new_word = word.replace(pattern, replacement)
        new_corpus[new_word] = corpus[word]
    return best_pair, new_corpus

def wordpiece_merge_step(corpus):
    """Finds best pair by mutual likelihood score: Count(u,v) / (Count(u)*Count(v))."""
    pairs = get_stats(corpus)
    if not pairs:
        return None, corpus
        
    # Count individual symbols
    symbol_counts = defaultdict(int)
    for word, freq in corpus.items():
        for s in word.split():
            symbol_counts[s] += freq
            
    best_pair = None
    best_score = -1.0
    for (u, v), pair_count in pairs.items():
        score = pair_count / (symbol_counts[u] * symbol_counts[v])
        if score > best_score:
            best_score = score
            best_pair = (u, v)
            
    # Execute merge
    new_corpus = {}
    pattern = ' '.join(best_pair)
    replacement = ''.join(best_pair)
    for word in corpus:
        new_word = word.replace(pattern, replacement)
        new_corpus[new_word] = corpus[word]
    return best_pair, new_corpus

# Toy Vocabulary Corpus
corpus_bpe = {'n e w': 5, 'n e w e s t': 6, 'w i d e': 3, 'w i d e s t': 4}
corpus_wp = corpus_bpe.copy()

best_bpe, _ = bpe_merge_step(corpus_bpe)
best_wp, _ = wordpiece_merge_step(corpus_wp)

print("--- Tokenization Merge Criterion Comparison ---")
print(f"BPE Selected Merge (Pure Frequency):  {best_bpe}")
print(f"WordPiece Selected Merge (Likelihood): {best_wp}")

### 3. Step-by-Step Code Walkthrough
1. **BPE Frequency Metric:** Merges the raw most frequent co-occurring character bigram.
2. **WordPiece Likelihood Metric:** Normalizes bigram frequency by individual constituent frequencies, prioritizing co-occurrences that exceed chance.
3. **Vocabulary Control:** Iteratively runs until the target vocabulary size is reached.

---

### 4. Industry Applications & Trade-offs
- **Modern LLMs:** Byte-level BPE (BBPE) eliminates unknown `[UNK]` tokens by operating on raw UTF-8 bytes.
- **Trade-off:** Greedy merge strategies do not guarantee optimal sentence tokenization length (addressed by Unigram/SentencePiece).

---

### 5. Top Interview Questions & Answers
**Q1: How does Byte-level BPE (BBPE) eliminate the `[UNK]` out-of-vocabulary problem?**
> **Answer:** BBPE defines its initial base vocabulary as all 256 possible bytes in UTF-8. Because any arbitrary character, symbol, or emoji decomposes into valid bytes, every possible string can be tokenized without unknowns.

**Q2: What is the "Fertility Rate" of a tokenizer?**
> **Answer:** The average number of subword tokens produced per word. A lower fertility rate indicates a more compact, efficient tokenizer representation.